In [8]:
import os
print("je m'execute depuis :", os.getcwd())
print("le dossier data existe :", os.path.isdir("../data"))
print("le fichier existe     :", os.path.isfile("../data/restaurants.json"))
print("contenu de ../data    :", os.listdir("../data"))

je m'execute depuis : c:\Users\Dubos\Documents\json-api-geocodage\notebooks
le dossier data existe : True
le fichier existe     : True
contenu de ../data    : ['restaurants.json']


In [9]:
import json

with open("../data/restaurants.json", encoding="utf-8") as f:
    data = json.load(f)

print(type(data))
print(data["ville"], "-", data["pays"])
print("nombre de restaurants :", len(data["restaurants"]))

<class 'dict'>
Catane - Italie
nombre de restaurants : 4


In [10]:
print(data["restaurants"][0]["nom"])
print(data["restaurants"][0]["notes"][0])

Gelateria Zio Pietro dal 1964
4.6


In [11]:
for r in data["restaurants"]:
    moyenne = sum(r["notes"]) / len(r["notes"])
    print(f"{r['nom']:32} {moyenne:.2f}")

Gelateria Zio Pietro dal 1964    4.70
A Casa d'Amici                   4.45
La Bitta                         4.15
Steak House                      4.00


In [12]:
print(json.dumps(data["restaurants"][1], ensure_ascii=False, indent=2))

{
  "nom": "A Casa d'Amici",
  "adresse": "Via Fischetti, 14, 95131 Catania CT",
  "type": "trattoria",
  "notes": [
    4.4,
    4.5
  ]
}


In [18]:
import pandas as pd

df = pd.json_normalize(data)
print(df.shape)
print(df.columns.tolist())

(1, 4)
['ville', 'pays', 'releve', 'restaurants']


In [19]:
df = pd.json_normalize(data, record_path="restaurants")
print(df)

                             nom                                   adresse  \
0  Gelateria Zio Pietro dal 1964  Via Porta di Ferro, 47, 95131 Catania CT   
1                 A Casa d'Amici       Via Fischetti, 14, 95131 Catania CT   
2                       La Bitta    Via Acquicella Porto, 95121 Catania CT   
3                    Steak House   Via Porta di Ferro, 8, 95100 Catania CT   

        type                 notes  
0    glacier       [4.6, 4.8, 4.7]  
1  trattoria            [4.4, 4.5]  
2    poisson  [4.2, 4.0, 4.3, 4.1]  
3      grill            [3.9, 4.1]  


In [20]:
df = pd.json_normalize(data, record_path="restaurants", meta=["ville", "pays"])
print(df)

                             nom                                   adresse  \
0  Gelateria Zio Pietro dal 1964  Via Porta di Ferro, 47, 95131 Catania CT   
1                 A Casa d'Amici       Via Fischetti, 14, 95131 Catania CT   
2                       La Bitta    Via Acquicella Porto, 95121 Catania CT   
3                    Steak House   Via Porta di Ferro, 8, 95100 Catania CT   

        type                 notes   ville    pays  
0    glacier       [4.6, 4.8, 4.7]  Catane  Italie  
1  trattoria            [4.4, 4.5]  Catane  Italie  
2    poisson  [4.2, 4.0, 4.3, 4.1]  Catane  Italie  
3      grill            [3.9, 4.1]  Catane  Italie  


In [25]:
import json

# Ouverture et lecture du fichier villes.json
with open("../data/villes.json", encoding="utf-8") as f:
    data_villes = json.load(f)

# 1. Nombre de villes
print("Nombre de villes :", len(data_villes["villes"]))

# 2. Nom de la deuxième ville (index 1)
deuxieme_ville = data_villes["villes"][1]["nom"]
print("Nom de la deuxième ville :", deuxieme_ville)

# 3. Nombre de restaurants à Syracuse (index 2)
nb_restaurants_syracuse = len(data_villes["villes"][2]["restaurants"])
print("Nombre de restaurants à Syracuse :", nb_restaurants_syracuse)

# 4. Nom du premier restaurant de Palerme (index 1)
nom_premier_resto_palerme = data_villes["villes"][1]["restaurants"][0]["nom"]
print("Nom du premier restaurant de Palerme :", nom_premier_resto_palerme)


Nombre de villes : 3
Nom de la deuxième ville : Palerme
Nombre de restaurants à Syracuse : 2
Nom du premier restaurant de Palerme : Antica Focacceria


In [26]:
for ville in data_villes["villes"]:
    nom_ville = ville["nom"]
    nb_restaurants = len(ville["restaurants"])
    print(f"{nom_ville} : {nb_restaurants} restaurants")

Catane : 3 restaurants
Palerme : 2 restaurants
Syracuse : 2 restaurants


In [27]:
import pandas as pd

# Transformation du JSON imbriqué en DataFrame
df_restaurants = pd.json_normalize(
    data_villes,
    record_path=["villes", "restaurants"],
    meta=[
        "pays",
        ["villes", "nom"],
        ["villes", "region"],
        ["villes", "habitants"]
    ]
)

# Affichage des dimensions et des premières lignes pour vérifier
print("Dimensions du DataFrame :", df_restaurants.shape)
print(df_restaurants)

Dimensions du DataFrame : (7, 7)
                             nom       type                 notes    pays  \
0  Gelateria Zio Pietro dal 1964    glacier       [4.6, 4.8, 4.7]  Italie   
1                 A Casa d'Amici  trattoria            [4.4, 4.5]  Italie   
2                       La Bitta    poisson  [4.2, 4.0, 4.3, 4.1]  Italie   
3              Antica Focacceria  trattoria            [4.3, 4.2]  Italie   
4               Ferro di Cavallo  trattoria            [4.5, 4.1]  Italie   
5                    Don Camillo    poisson  [4.7, 4.8, 4.6, 4.9]  Italie   
6               Gelateria Bianca    glacier                 [4.2]  Italie   

  villes.nom villes.region villes.habitants  
0     Catane        Sicile           298762  
1     Catane        Sicile           298762  
2     Catane        Sicile           298762  
3    Palerme        Sicile           630828  
4    Palerme        Sicile           630828  
5   Syracuse        Sicile           121605  
6   Syracuse        Sicile  

In [31]:
# 1. Ajouter la colonne note_moyenne arrondie à deux décimales
df_restaurants["note_moyenne"] = df_restaurants["notes"].apply(
    lambda x: round(sum(x) / len(x), 2)
)

# 2. Quelle ville a la meilleure note moyenne ?
# On groupe par nom de ville et on calcule la moyenne des notes
meilleure_ville = (
    df_restaurants.groupby("villes.nom")["note_moyenne"].mean().idxmax()
)
print("Ville avec la meilleure note moyenne :", meilleure_ville)

# 3. Combien de restaurants de chaque type, tous lieux confondus ?
compte_types = df_restaurants["type"].value_counts()
print("\nNombre de restaurants de chaque type :")
print(compte_types)

# Affichage du DataFrame mis à jour pour vérifier
display(df_restaurants[["villes.nom", "nom", "type","villes.region", "villes.habitants", "note_moyenne"]])

Ville avec la meilleure note moyenne : Syracuse

Nombre de restaurants de chaque type :
type
trattoria    3
glacier      2
poisson      2
Name: count, dtype: int64


,villes.nom,nom,type,villes.region,villes.habitants,note_moyenne
0,Catane,Gelateria Zio Pietro dal 1964,glacier,Sicile,298762,4.70
1,Catane,A Casa d'Amici,trattoria,Sicile,298762,4.45
2,Catane,La Bitta,poisson,Sicile,298762,4.15
3,Palerme,Antica Focacceria,trattoria,Sicile,630828,4.25
4,Palerme,Ferro di Cavallo,trattoria,Sicile,630828,4.30
5,Syracuse,Don Camillo,poisson,Sicile,121605,4.75
6,Syracuse,Gelateria Bianca,glacier,Sicile,121605,4.20


In [32]:
import requests

url = "https://api-adresse.data.gouv.fr/search/"
r = requests.get(url, params={"q": "20 avenue de Ségur 75007 Paris", "limit": 1})

print(r.status_code)

200


In [33]:
reponse = r.json()
print(type(reponse))
print(reponse.keys())

<class 'dict'>
dict_keys(['type', 'features', 'query'])


In [34]:
import requests

def interroger(url, params, headers=None):
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.Timeout:
        print("Trop lent, on abandonne cette requête")
    except requests.exceptions.HTTPError as e:
        print("Le serveur a répondu une erreur :", e)
    except requests.exceptions.RequestException as e:
        print("Problème réseau :", e)
    return None

In [35]:
print(interroger("https://api-adresse.data.gouv.fr/search/", {"q": "Lille", "limit": 1}) is not None)
print(interroger("https://api-adresse.data.gouv.fr/nimportequoi", {"q": "Lille"}))

True
Le serveur a répondu une erreur : 404 Client Error: Not Found for url: https://api-adresse.data.gouv.fr/nimportequoi?q=Lille
None


In [40]:
import requests

adresses = [
    "20 avenue de Ségur 75007 Paris",
    "Grand Place, Lille",
    "1 rue de la Paix, Paris",
    "place Bellecour, Lyon",
    "zzzzz qwerty 99999",
]

url = "https://api-adresse.data.gouv.fr/search/"

# Test sur la première adresse
params = {"q": adresses[0], "limit": 1}
reponse = requests.get(url, params=params)

print("Code de statut :", reponse.status_code)

data = reponse.json()
if data["features"]:
  premier_resultat = data["features"][0]["properties"]
  print("Label :", premier_resultat.get("label"))
  print("Score :", premier_resultat.get("score"))

Code de statut : 200
Label : 20 Avenue de Ségur 75007 Paris
Score : 0.9716281818181818


In [41]:
import pandas as pd

resultats = []

for adresse in adresses:
  params = {"q": adresse, "limit": 1}
  reponse = requests.get(url, params=params)

  # Dictionnaire par défaut avec des valeurs vides au cas où l'adresse échoue
  info_adresse = {
      "adresse_saisie": adresse,
      "label": None,
      "score": None,
      "postcode": None,
      "city": None,
  }

  if reponse.status_code == 200:
    data = reponse.json()
    # On vérifie si l'API a bien trouvé des résultats
    if data["features"]:
      props = data["features"][0]["properties"]
      info_adresse["label"] = props.get("label")
      info_adresse["score"] = props.get("score")
      info_adresse["postcode"] = props.get("postcode")
      info_adresse["city"] = props.get("city")

  resultats.append(info_adresse)

# Transformation en DataFrame pour un affichage propre et lisible
df_resultats = pd.DataFrame(resultats)
display(df_resultats)

,adresse_saisie,label,score,postcode,city
0,20 avenue de Ségur 75007 Paris,20 Avenue de Ségur 75007 Paris,0.971628,75007,Paris
1,"Grand Place, Lille",Place de Gand 59800 Lille,0.568244,59800,Lille
2,"1 rue de la Paix, Paris",1 Rue de la Paix 75002 Paris,0.964060,75002,Paris
3,"place Bellecour, Lyon",Place Bellecour 69002 Lyon,0.967367,69002,Lyon
4,zzzzz qwerty 99999,NaN,NaN,NaN,NaN


Combien d'adresses ont été trouvées ? > 4 adresses sur les 5 ont été trouvées. La dernière (zzzzz qwerty 99999) renvoie des valeurs manquantes (None).Laquelle a le score le plus bas parmi celles trouvées, et pourquoi à ton avis ? > Les lieux dits ou grandes places (comme "Grand Place, Lille" ou "place Bellecour, Lyon") obtiennent généralement un score légèrement plus bas que les adresses précises avec numéro et rue, car l'API cherche une zone ou un centroïde plutôt qu'un point exact.
Le label renvoyé correspond-il toujours à ce que tu avais demandé ? > Pas exactement mot pour mot. L'API normalise et standardise le format (en ajoutant par exemple la commune, le code postal officiel ou en modifiant la casse) pour correspondre à sa base de données officielle.  